<!--
---
PURPOSE: Discover sessions, locate NWB files, and summarize modalities.
REQUIRES:
  - outputs/reports/config_snapshot.json
PRODUCES:
  - outputs/reports/session_inventory.parquet
  - outputs/reports/session_{id}_summary.html
  - outputs/reports/artifact_registry.parquet
WHAT_NEXT: notebooks/02_Neural_Data_Spikes_and_Events.ipynb
---
-->

# 01 Session Discovery and Metadata

**Purpose**
Discover sessions, locate NWB files, and summarize modalities.

**Requires**
- `outputs/reports/config_snapshot.json`

**Produces**
- `outputs/reports/session_inventory.parquet`
- `outputs/reports/session_{id}_summary.html`
- `outputs/reports/artifact_registry.parquet`

**What to run next**
- `notebooks/02_Neural_Data_Spikes_and_Events.ipynb`

We validate session inputs, inspect modalities, and generate an inventory.


## Environment Setup
We add the repo `src/` to the Python path so notebooks can import shared modules.

In [1]:
import sys
from pathlib import Path
ROOT = Path.cwd()
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

## Prerequisite Check
We parse the notebook header and validate required artifacts before running downstream steps.

In [3]:
from pathlib import Path
from reports import parse_notebook_header, validate_prerequisites

nb_path = Path.cwd() / "notebooks" / "01_Session_Discovery_and_Metadata.ipynb"
header = parse_notebook_header(nb_path)
missing = validate_prerequisites(header.get("REQUIRES", []))
if missing:
    print("Missing prerequisites:")
    for item in missing:
        print(" -", item)
else:
    print("All prerequisites satisfied.")

ModuleNotFoundError: No module named 'reports'

## Step 1: Load sessions list
If sessions.csv is missing, a template is generated from sessions.txt.

In [ ]:
from io_sessions import load_sessions_csv

sessions = load_sessions_csv()
sessions.head()

## Step 2: Inspect modalities
We open each session (without downloading large video assets) and record available modalities.

In [ ]:
from io_sessions import get_session_bundle
import pandas as pd

SESSION_IDS = sessions.session_id.tolist()
records = []
for session_id in SESSION_IDS:
    bundle = get_session_bundle(session_id, sessions)
    rec = {"session_id": session_id, "nwb_path": str(bundle.nwb_path) if bundle.nwb_path else ""}
    rec.update(bundle.modalities_present)
    records.append(rec)

inventory = pd.DataFrame(records)
inventory.head()

## Step 3: Save inventory and per-session summaries
These files are used by downstream notebooks and the end-to-end pipeline.

In [ ]:
from timebase import write_parquet_with_timebase
from config import get_config, make_provenance
import pandas as pd

cfg = get_config()
outputs = cfg.outputs_dir / "reports"
outputs.mkdir(parents=True, exist_ok=True)

inv_path = outputs / "session_inventory.parquet"
write_parquet_with_timebase(
    inventory,
    inv_path,
    provenance=make_provenance(None, "nwb"),
)

for _, row in inventory.iterrows():
    html_path = outputs / f"session_{int(row['session_id'])}_summary.html"
    row.to_frame().to_html(html_path)

inv_path

## Step 4: Render artifact registry
This gives a quick overview of what exists and where.

In [ ]:
from reports import write_artifact_registry

registry_path = write_artifact_registry()
registry_path